# DAT408 · Explainable AI agents: decision evidence with DynamoDB and Neptune

AnyCompany Home Loans underwrites FHA mortgages with an AI agent. For each application it pulls the
credit report, verifies income, works out debt-to-income and loan-to-value, looks up the rules in the
FHA handbook, HUD 4000.1, and recommends **approve**, **deny**, or **escalate** to a human
underwriter. It runs in Amazon Bedrock AgentCore Runtime, so every run is traced and timed, and the
dashboards are green.

One of its recommendations was to deny **Carlos Salazar**, application LA-2026-00148: a \$280,000 loan
on a \$285,000 house. Months later, a fair-lending examiner writes:

> *For application LA-2026-00148, state (1) the specific reasons for denial, (2) the information the
> system relied on, and (3) the policies it applied. (4) Show that applicants with similar profiles
> were treated consistently.*

The dashboards can tell the examiner that the agent ran, how long it took, and that nothing failed.
They can't say what it read, which edition of a policy it saw, or why it decided. Answering takes
**decision evidence**: what the agent was given and what it said, written once, as it decides, and
stored so that we can also answer the questions nobody has asked yet.

In this notebook, we'll build that evidence the way a database engineer would, one design decision at
a time: **what to record**, **how to key it**, **what goes in a graph**, and **when to check what the
agent claims**. Each decision has a few options. For each one, we'll make a prediction, run the option
(all the code is already here), and see what it costs.

**Keep the guide open beside this notebook.** Each section here has a page in the Workshop Studio
guide, in the other tab, with the question, the options, and what to look for. Each section heading
below names its page.

*AnyCompany Home Loans, its applicants, and their decisions are fictional. FHA and HUD are real: the
policy texts are HUD's published handbook and federal regulations, used here as sample data.*

![What runs in your account: the notebook invokes the agent in AgentCore Runtime, which searches the Knowledge Base and writes each step to DynamoDB. DynamoDB Streams feed a Lambda transform that projects each decision into Neptune Analytics. Telemetry goes to AgentCore Observability; the MCP server reads DynamoDB and the graph.](images/architecture.svg)

**How to run it.** Run the cells in order: select a cell and press **Shift+Enter**.
If the kernel restarts, run Cell 0.1 again: it reconnects and finds your Carlos evaluation.
For a runnable reference to the openCypher in Decisions 3 and 4, open **04 openCypher reference.ipynb**
in this folder.

## Start · A decision to answer for

*Guide page: **Start**.*

To explain a decision, we first need one. We'll start by running the agent three times at once:
Carlos's application once, and **Saanvi Sarkar**'s, LA-2026-00149, twice. Your run of Carlos stands in
for the denial the examiner is asking about.

Saanvi's case is the interesting one. Her back-end debt-to-income ratio is 49.98%, just under the 50%
that HUD 4000.1 allows only with two documented compensating factors, such as cash reserves. **Before
you run Cell 0.2, make a guess: will her two runs agree?**

**Cell 0.1 · Connect.** This cell calls `connect()`, which finds this stack's resources (the DynamoDB
table, the Neptune Analytics graph, and the agent's runtime), gives them short names, and sets up
`%%oc`, the notebook's graph view. It then deploys the stream transform, a Lambda function on the
table's stream. From here on, each step the agent writes to DynamoDB reaches the graph a few seconds
later. We should see *Setup complete*, followed by *Deployed transform* with the stream trigger
*Enabled*. After a kernel restart, this cell also finds your Carlos evaluation, so it's the only cell
to run again.

*Under the hood:* `connect` in `agent/session.py` reads the stack's CloudFormation outputs and binds
the names the later cells use. `deploy_reference_transform` uploads `agent/transform.py` as the
`dat408-stream-processor` Lambda's code. That file is the stream transform: it turns each step item
from the stream into nodes and edges, merged on ids taken from the record, so a replay adds nothing.
To read either function here, run `connect??` or `deploy_reference_transform??` in a new cell (two question marks: Jupyter shows a function's source).

In [ ]:
# MIT No Attribution
#
# Copyright 2026 Amazon.com, Inc. or its affiliates. All Rights Reserved.
#
# Permission is hereby granted, free of charge, to any person obtaining a copy of this
# software and associated documentation files (the "Software"), to deal in the Software
# without restriction, including without limitation the rights to use, copy, modify,
# merge, publish, distribute, sublicense, and/or sell copies of the Software, and to
# permit persons to whom the Software is furnished to do so.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR IMPLIED,
# INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY, FITNESS FOR A
# PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT
# HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, WHETHER IN AN ACTION
# OF CONTRACT, TORT OR OTHERWISE, ARISING FROM, OUT OF OR IN CONNECTION WITH THE
# SOFTWARE OR THE USE OR OTHER DEALINGS IN THE SOFTWARE.
#
# SPDX-License-Identifier: MIT-0
# Cell 0.1 · connect, and deploy the stream transform
from agent.session import connect
connect(globals(), graph=True)
deploy_reference_transform()

**Cell 0.2 · Run the agent.** The three runs happen in parallel, so this takes under a minute. Each run
is its own evaluation and writes its steps to DynamoDB as it goes. The first table shows each run's
recommendation next to the facts it's based on. Below that are 20 runs of Saanvi's application that we
recorded earlier on this same build of the agent. The last line sets `CARLOS`, the evaluation that
every later cell reads.

*Under the hood:* `run_evaluations` in `agent/session.py` invokes the agent in AgentCore Runtime once
per application, in parallel. The code that runs inside the runtime is `agent/runtime_app.py`, and its
`_emit_trace_event` writes the evidence. It writes a `META` header first, then one `STEP#` item per
step with `attribute_not_exists(PK)`, so nothing is ever overwritten. Each tool's full output goes to
S3, with its SHA-256 on the item, and the final decision goes in one `TransactWriteItems` call. To read
the notebook side, run `run_evaluations??` in a new cell (two question marks: Jupyter shows the function's source); for the runtime side, open
`agent/runtime_app.py`.

In [ ]:
# Cell 0.2 · Carlos once and Saanvi twice, in parallel (under a minute)
runs = run_evaluations(
    ["LA-2026-00148", "LA-2026-00149", "LA-2026-00149"])
show_recorded_runs("LA-2026-00149")
CARLOS = choose_carlos(runs)

Your two runs of Saanvi may agree on the outcome, and they may even take the same steps. On this
build, 20 recorded runs all escalated, but they read 6 different sets of passages: the outcome held,
while what the agent relied on changed. Nothing in the request pins down either one. The model, Claude
Sonnet 5, samples at its default temperature, and a new prompt or agent build can move both. Each
header records the model, prompt, and build that ran.

That's the point of this first exercise: we can't re-run a decision later to explain it. The
explanation has to come from what was recorded at the time, and that's where Decision 1 begins.

## Decision 1 · What to record

*Guide page: **1 · What to record**.*

The examiner's first three asks are about a single decision: the reasons, the information relied on,
and the policies applied. In this section, we'll compare three ways to keep what it takes to answer
them:

| Option | What it keeps | What it costs |
|---|---|---|
| **A · Telemetry** | each run's health, from AgentCore Observability | one retention per log group; nothing about what the agent read |
| **B · A transcript, explained on request** | everything the agent saw and wrote; a model explains it when someone asks | a model call per answer, and new wording each time |
| **C · Decision evidence** | each tool result, each search's policies and edition, the decision's citations, written once as structured items | a schema, and a write per step |

**Which of these answers asks 1 to 3 the same way every time someone asks?**

**Cell 1.1 · Option A: telemetry.** This is what we get without writing any code: AgentCore
Observability's view of your runs over the last hour, from CloudWatch and X-Ray. It shows
invocations, latency, errors, and throttles. Metrics arrive about a minute after a run, so if the cell
shows none, wait a minute and run it again.

*Under the hood:* this cell calls `show_agentcore_observability` in `agent/workshop.py`. It reads the
runtime's CloudWatch metrics in the `AWS/Bedrock-AgentCore` namespace with `GetMetricData`, and its
X-Ray trace summaries with `GetTraceSummaries`. We wrote none of this: AgentCore emits it for every
invocation. To read the code, run `show_agentcore_observability??` in a new cell (two question marks: Jupyter shows the function's source).

In [ ]:
# Cell 1.1 · option A: telemetry, over the last 60 minutes
show_agentcore_observability(minutes=60)

Telemetry is the right tool for answering *is the agent healthy?*, but it can't answer any of the
examiner's asks. Nothing in it says which facts the agent used or which policy it applied. It also
keeps everything for as long as its log group does. A log group has one retention setting, so it
can't keep a denial for 7 years and an approval for 25 months, or put a hold on one case for an audit.

**Cell 1.2 · Option B: explain it later.** This cell gives the transcript approach its best possible
chance. It reads every step's complete output back from S3, checks it against its hash, and hands it
to the agent's own model with a compliance analyst's prompt, twice. Under each summary, a fixed
checker (code, not a model) marks ⚠️ on any reason that states a fact the record doesn't hold, and ℹ️
on a risk factor presented as a reason. Read the two summaries side by side.

*Under the hood:* this cell calls `summarize_decision` in `agent/session.py`. It reads Carlos's
partition, fetches each step's full output from S3 and checks its hash, joins the steps into one
transcript, and sends that transcript to the agent's model with the Bedrock `Converse` API, once per
summary. To read the code, run `summarize_decision??` in a new cell (two question marks: Jupyter shows the function's source).

In [ ]:
# Cell 1.2 · option B: the agent's model explains the denial, twice
summaries = summarize_decision(CARLOS, runs=2)

Both summaries are plausible, but they aren't the same. Below them is the count from 20 summaries of
one recorded evaluation: they gave 14 different sets of reasons. None of them stated a fact the record
doesn't hold, so the problem isn't that the model makes things up. The problem is that every
explanation is a new one. Regulation B asks for the reasons that were actually considered, and an
explanation written afterward can't show that. Only the record can.

**Cell 1.3 · Option C: decision evidence.** This is the record the runtime writes while the agent
works. Each evaluation gets its own partition, `EVAL#<id>`. The header and every step are items in that
partition, so a single Query returns the whole decision, in order:

<p><b>dat408-agent-traces: one partition per evaluation.</b> The agent writes an item at every step. One Query on PK = <code>EVAL#&lt;id&gt;</code> returns the header and every step, in order.</p>
<table style="border-collapse:collapse;font-size:13px;width:100%"><tr><th style="background:#2d5bba;color:#fff;text-align:left;padding:6px 10px">PK</th><th style="background:#2d5bba;color:#fff;text-align:left;padding:6px 10px">SK</th><th style="background:#2d5bba;color:#fff;text-align:left;padding:6px 10px">attributes</th></tr><tr><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#2d5bba;font-weight:600">EVAL#&lt;id&gt;</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;font-family:monospace;font-weight:600">META</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9">the header: status · model_id · prompt_version · agent_build · kb_ingestion_job_id · started_at · step_timestamp · outcome</td></tr><tr><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#2d5bba;font-weight:600"></td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;font-family:monospace;font-weight:600">STEP#000</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9">credit report: action · action_input · thought · credit_score 618 · derogatory_marks 2</td></tr><tr><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#2d5bba;font-weight:600"></td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;font-family:monospace;font-weight:600">STEP#001</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9">income check: verified_annual_income 45,000</td></tr><tr><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#2d5bba;font-weight:600"></td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;font-family:monospace;font-weight:600">STEP#002</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9">debt-to-income: front_end_dti_pct · back_end_dti_pct 58.9</td></tr><tr><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#2d5bba;font-weight:600"></td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;font-family:monospace;font-weight:600">STEP#003</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9">property value: ltv_pct 98.2</td></tr><tr><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#2d5bba;font-weight:600"></td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;font-family:monospace;font-weight:600">STEP#004–006</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9">policy searches: retrieved_policy_ids · retrieved_policy_versions (the edition) · retrieved_chunk_ids</td></tr><tr><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#2d5bba;font-weight:600"></td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;font-family:monospace;font-weight:600">STEP#007</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9">the decision: outcome · reasoning · policies_cited · policies_violated · adverse_action_reasons · risk_factors</td></tr><tr><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#2d5bba;font-weight:600"></td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;font-family:monospace;font-weight:600"></td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#6b6a65">bands (credit_band · dti_band · ltv_band) · citation_check · ungrounded_month, only if the check fails</td></tr><tr><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;color:#2d5bba;font-weight:600">POLICY#&lt;id&gt;</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9;font-family:monospace;font-weight:600">MONTH#2026-10</td><td style="text-align:left;padding:5px 10px;border-top:1px solid #e1e0d9">a running count: decisions_citing · decisions_failing, added in the decision's transaction</td></tr></table>
<p style="color:#6b6a65;font-size:12px">Every step also keeps observation_sha256 and observation_s3_uri: the tool's full output in S3, hash-checked. Approvals expire (TTL) after 25 months, everything else after 7 years.</p>

<p><b>Four global secondary indexes, each for a question you can name</b></p>
<table style="border-collapse:separate;border-spacing:8px;font-size:13px;width:100%"><tr><td style="vertical-align:top;text-align:left;border:1px solid #c3c2b7;border-radius:8px;padding:10px 14px;width:50%"><span style="font-family:monospace;font-weight:700;color:#2d5bba">application-index</span><br>partition key application_id<br>sort key step_timestamp<br>projects keys only<br><i style="color:#5a3fc0">Which evaluations ran for LA-2026-00148?</i></td><td style="vertical-align:top;text-align:left;border:1px solid #c3c2b7;border-radius:8px;padding:10px 14px;width:50%"><span style="font-family:monospace;font-weight:700;color:#2d5bba">band-index</span><br>partition key credit_band + dti_band + ltv_band<br>sort key step_timestamp<br>projects outcome, compensating_factors, policies_violated<br><i style="color:#5a3fc0">Were applicants like Carlos decided alike?</i></td></tr><tr><td style="vertical-align:top;text-align:left;border:1px solid #c3c2b7;border-radius:8px;padding:10px 14px;width:50%"><span style="font-family:monospace;font-weight:700;color:#2d5bba">ungrounded-index</span><br>partition key ungrounded_month<br>sort key step_timestamp<br>projects application_id, outcome, ungrounded_citations<br><i style="color:#5a3fc0">Which new decisions failed the citation check?</i></td><td style="vertical-align:top;text-align:left;border:1px solid #c3c2b7;border-radius:8px;padding:10px 14px;width:50%"><span style="font-family:monospace;font-weight:700;color:#2d5bba">prompt-index</span><br>partition key prompt_version<br>sort key step_timestamp<br>projects keys only<br><i style="color:#5a3fc0">Which decisions ran under one prompt?</i></td></tr></table>
<p style="color:#6b6a65;font-size:12px">band-index keys on all three bands at once. Both sparse indexes hold only items with their key: decisions that failed the check, and headers that record a prompt.</p>

The cell reads Carlos's partition with a strongly consistent Query and shows what each tool returned
and which edition each search read. It then checks every step's hash against its full output in S3,
and reads everything a second time to show that the answer doesn't change.

*Under the hood:* this cell calls `show_evidence` in `agent/session.py`. The part that matters is one
`Query` on `PK = EVAL#<id>` with `ConsistentRead=True`; the rest formats the output and checks the
hashes. To read it without leaving the notebook, run `show_evidence??` in a new cell (two question marks: Jupyter shows the function's source).

In [ ]:
# Cell 1.3 · option C: the decision evidence, read back by key, twice
show_evidence(CARLOS)

With one Query, we've answered asks 1 to 3 from recorded facts, for about 6 read units, and that cost
stays the same at any table size. We have the policies the decision says Carlos failed (II.A.2.b on
loan-to-value and II.A.5.d on debt-to-income), the facts behind them, and the edition each search
read. The record also keeps Carlos's name and the property's address, where Regulation B
wants them. This demo agent sees both, the name in its first prompt and the address from the property
check; in production, we'd keep them out of the model's input.

**Rule 1: record what it read.**

Next, we'll decide how to find this evidence again, for Carlos and for the 3,000,000 past decisions in
the table.

## Decision 2 · How to key it

*Guide page: **2 · How to key it**.*

Finding one decision is easy, because its partition key is its evaluation. The examiner's fourth ask
isn't about one decision, though: it's about "applicants with similar profiles". And a compliance team
will ask more questions than anyone can list today.

In DynamoDB, each question needs a way in. In this section, we'll sort five questions into four kinds
of access. A question can use the table's own **key**. It can use an **index**, a global secondary
index (GSI) that we can add at any time and that DynamoDB backfills. It can use a **count** that's
kept current in the same transaction that records each decision. Or it can wait until **later**, when
nothing holds the answer yet and we have to answer it some other way. The bottom of the table diagram
above shows the four indexes this table has.

**Cell 2.1 · Your sort.** Here are five questions. For each one, decide where its answer comes from:
`"key"`, `"index"`, `"count"`, or `"later"`. Type your answers into `ANSWERS` and run the cell. It
explains each answer, whether it's right or wrong, so there's no cost to guessing. Here's a hint: one
of the questions has no attribute that holds its answer.

In [ ]:
# Cell 2.1 · your sort: "key", "index", "count", or "later"
ANSWERS = {
    1: "",  # What did the agent do for Carlos, step by step?
    2: "",  # Evaluations of LA-2026-00148, and which model and prompt?
    3: "",  # How many decisions failed HUD 4000.1 II.A.5.d each month?
    4: "",  # Were applicants like Carlos decided the same way?
    5: "",  # Which denials cite a policy the agent never retrieved?
}
check_sort(ANSWERS)

**Cell 2.2 · Your sort, checked.** In Cell 2.1, you decided where each question's answer comes from.
Now we'll check those choices against the real table: 3,000,000 past decisions. If the design is right,
every question we keyed costs a few read units, however big the table gets, and the one we couldn't
key costs a full Scan.

The table lists the five questions, your choice beside the design's, what each one cost in read units,
and the answer it got. The chart shows the same read units, one bar per question, on a log scale: blue
for a key, an index, or a count, and red for question 5, which has no key.

*Under the hood:* this cell calls `run_questions` in `agent/session.py`. It prints the request behind
each answer first: a Query on the table's key, a Query on `application-index`, a Query on the
`POLICY#` count items, and a Query on `band-index` with all three bands. Question 5 has no key, so
it's priced as a Scan and not run. To read the code, run `run_questions??` in a new cell (two question marks: Jupyter shows the function's source).

In [ ]:
# Cell 2.2 · each question, answered the way the design answers it
run_questions(CARLOS)

Questions 1 to 3 cost single-digit read units, and they'd cost the same at a billion decisions.
Question 4 is one `band-index` Query: Carlos's past peers were all denied, because they all fail two
hard limits.

Question 5 is different. It compares what one item retrieved with what another item cited, and no key
holds that, so the cell prices it instead of running it: a Scan of the whole table, plus the join code
we'd have to write. For new decisions, the runtime marks a failed citation check with
`ungrounded_month`, and a sparse index on that attribute finds them by key. For the past decisions,
we'll use the graph, in Decision 4.

**Cell 2.2a · ✏️ Question 4 for Saanvi.** For Carlos, question 4 has an easy answer. Saanvi is where it
gets interesting: her back-end DTI sits just under the 50% limit, where HUD 4000.1 lets the outcome
depend on documented compensating factors. Now we'll write the `band-index` Query ourselves.

**What you write:** the `KeyConditionExpression` for Saanvi's three bands. `band-index` has a partition
key made of three attributes: `credit_band`, `dti_band` and `ltv_band`. The cell sets her values as
`credit`, `dti` and `ltv`.

**When it's right,** we see *✅ Your Query matches the reference*, then her past peers split by
compensating factors and outcome. If it isn't right yet, the cell says why and runs the reference
Query, so the rest of the page still works.

*Under the hood:* `show_peers` in `agent/session.py` runs your Query, follows `LastEvaluatedKey` to the
last page, and compares the decisions it found with the reference Query's. To read it, run
`show_peers??` in a new cell (two question marks:
Jupyter shows the function's source).

In [ ]:
# Cell 2.2a · ✏️ question 4 for Saanvi: her past peers on band-index
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Query band-index for every decision in Saanvi's three bands.
#    Change only the line marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
from boto3.dynamodb.conditions import Key
credit, dti, ltv = band_keys("LA-2026-00149")   # her bands, from her latest decision
PEERS = dict(
    IndexName="band-index",
    KeyConditionExpression=None,   # ✏️ the key condition for her three bands
)
show_peers("LA-2026-00149", PEERS)

<details>
<summary>🔑 Reveal solution</summary>

```python
PEERS = dict(
    IndexName="band-index",
    KeyConditionExpression=Key("credit_band").eq(credit) & Key("dti_band").eq(dti) & Key("ltv_band").eq(ltv),
)
```

</details>

Files with two documented compensating factors were mostly approved, which is what HUD 4000.1 allows
at 40/50. Files with none were split between denied and escalated. Those are similar files, decided two
ways, and that's the inconsistency the examiner's fourth ask is looking for. Because we planned the
bands, one index Query finds it.

**Rule 2: key what you know.**

**Cell 2.3 · A count must survive a resend.** The monthly policy counts behind question 3 are kept
current as each decision is written. But streams and retries deliver *at least* once, so the same
decision can arrive twice. This cell delivers one decision twice, first through three separate writes,
and then through the single transaction the runtime uses.

*Under the hood:* `record_decision` comes from `agent/reference.py`, and it's a single
`TransactWriteItems` call. It puts the decision item with `attribute_not_exists(PK)`, moves the header
from `IN_PROGRESS` to `COMPLETE`, and adds the monthly counts. If any condition fails, none of it is
written. The runtime records decisions the same way, in `_emit_trace_event` in `agent/runtime_app.py`.
`check_decision_transaction` in `agent/workshop.py` delivers the decision twice through each design
and compares what's left. To read the transaction, run `record_decision??` in a new cell (two question marks: Jupyter shows the function's source).

In [ ]:
# Cell 2.3 · one decision delivered twice, through two designs
record_decision = reference_answer("record_decision")   # the runtime's
check_decision_transaction(record_decision)

The separate writes count the resent decision twice. The transaction counts it only once. Its
condition, that the decision item mustn't exist yet, fails on the resend, so DynamoDB cancels the
whole transaction, count and all.

That covers the questions we can name ahead of time. Next, we'll look at a question no key can answer.

## Decision 3 · What goes in the graph

*Guide page: **3 · What goes in the graph**.*

Question 5 had no key, and here's a harder question still. An audit finds that **doc-extract 2.3.1**,
the model version that read applicants' income documents from May 4 to 24, misread some of them.
Which decisions, and which adverse action notices, rest on what it read, either directly or through
anything built from it?

*Built from* is the hard part. The employer's name and EIN that an extraction reads become an employer
record, which is refreshed every month, and every later applicant at that employer has their
employment checked against it. An applicant who re-applies reuses their old documents. So a date filter misses everything after May 24, and at the same time catches thousands of
decisions that never touched 2.3.1. To answer the question, we need to follow what each result was
built from, for as long as the trail goes.

This history includes that trail. It's a synthetic multi-agent build that records, for each artifact,
where its amounts came from (`DERIVED_FROM`) and where its employer's identity came from
(`IDENTITY_FROM`). The trail sits in the graph next to what decisions share:

![The evidence graph. Left, what decisions share: Application, Decision, Policy (retrieved, cited, violated), Chunk (passages read), Tool, Band. Right, the lineage: Version, extraction, employer record, income, ratios, Decision and Notice, linked by PRODUCED_BY, DERIVED_FROM, IDENTITY_FROM, USED, FOR and RESTS_ON.](images/graph-model.svg)

*Both halves of this picture are option B, the graph this workshop runs: on the left, what decisions
share; on the right, what each result was built from. Option A would also add every step as a node.*

In this section, we'll weigh three options. Option **A** puts every record in the graph, steps
included. Option **B** puts only what decisions share, plus the lineage, in the graph, and leaves the
steps in DynamoDB. Option **C** uses no graph at all: it exports the same edges to S3 and walks them
with Athena's recursive SQL. This workshop runs option B.

**Before you run the cells, make a guess: after the fix, which reached more decisions, the amounts
2.3.1 read or the employer identities?**

**Cell 3.1 · One pay stub, 31 decisions.** Before we walk the lineage across 3 million decisions, let's
zoom in on one document that 2.3.1 read. On May 6, it read an applicant's pay stubs, and the employer's
name and EIN it extracted became employer E000676's record. The record was refreshed every month, and
every later applicant at that employer had their employment checked against it. Their income came from
their own documents; the employer's identity came from what 2.3.1 read. This cell sends the openCypher below to Neptune Analytics and draws what
comes back, with 2.3.1 on the left and the decisions on the right.

- The first `MATCH` goes from the model version (`$source`, doc-extract 2.3.1) to the pay-stub
  extraction it produced (`PRODUCED_BY`), and to the employer record that took its identity from it
  (`IDENTITY_FROM`).
- The second follows `IDENTITY_FROM` backward, through the monthly refreshes, to each applicant's
  income result, and then to the decision that used it (`USED`). `*1..6` is a variable-length
  pattern: one query, however many refreshes there were.
- The third adds the adverse action notice each of those decisions sent (`FOR`). `UNION` joins the
  paths.
- `$source` comes from `params`, which Cell 0.2 set. Decisions are colored by outcome and labeled with
  their day.

*Under the hood:* `show_from_source` in `agent/session.py` runs the query with `%%oc`, Neptune's graph
view, laid out from the source on the left; the relationships point back to what each result was built
from. To read it, run `show_from_source??` in a new cell (two question marks: Jupyter shows the function's source).

The view opens on *Console*: choose **Graph**. Click a node to see all its properties.

In [ ]:
# Cell 3.1 · one pay stub that doc-extract 2.3.1 read, and every decision that rests on it
ONE_PAY_STUB = """
MATCH p = (:Version {`~id`: $source})<-[:PRODUCED_BY]-(:Artifact)
          <-[:IDENTITY_FROM]-(:Artifact {`~id`: 'art:employer_record:E000676:2026-05-06'})
RETURN p
UNION
MATCH p = (:Artifact {`~id`: 'art:employer_record:E000676:2026-05-06'})
          <-[:IDENTITY_FROM*1..6]-(:Artifact {kind: 'income'})<-[:USED]-(:Decision)
RETURN p
UNION
MATCH (:Artifact {`~id`: 'art:employer_record:E000676:2026-05-06'})
      <-[:IDENTITY_FROM*1..6]-(:Artifact {kind: 'income'})<-[:USED]-(d:Decision)
MATCH p = (:Notice)-[:FOR]->(d)
RETURN p
"""
show_from_source(ONE_PAY_STUB)

One pay stub, read on May 6, sits under 31 decisions about 31 different applicants, made from May 6 to
September 28: each of them had their employment checked against the employer identity 2.3.1 read. 27 of them came after 2.3.1 was fixed on May 24, so a date filter would miss them, and 8
were denials that sent an adverse action notice. Their shared employer doesn't find them either: 53
decisions used one of E000676's records, and only these 31 rest on what 2.3.1 read. Only the edges say
which.

That's option B: what decisions share and what each result was built from go in the graph, and the
steps stay in DynamoDB, where one Query returns them in order. Cell 3.2 runs this walk for every
document 2.3.1 read.

**Cell 3.2 · ✏️ Walk back from the version.** This query runs a breadth-first search (BFS) in Neptune
Analytics. It starts at the model version, follows the edges listed in `edgeLabels` backward, and
visits each node once, counting how many hops away it is. As written, it follows employer identities
(`IDENTITY_FROM`) and nothing else.

**What you write:** first, run the cell as it is, and compare your walk's row with option C's. Then add
the edge label that follows amounts to `edgeLabels` (the graph model above names it), and run the cell
again.

**When it's right,** we see *✅ Your walk follows both routes*. Beside your walk, the table shows the
same walk over amounts only and over identities only, a date filter, a single hop (what a GSI on the
version would give us), and option C's recursive SQL.

*Under the hood:* `show_lineage` in `agent/session.py` runs your `WALK` through `neptune_query`, and
runs it again with its `edgeLabels` set to amounts only, identities only, and both, to check yours. The
date filter, one-hop, and recursive SQL rows are expected counts, computed when the stack deployed. To
read the code, run `show_lineage??` in a new cell (two question marks: Jupyter shows the function's source).

In [ ]:
# Cell 3.2 · ✏️ which decisions rest on doc-extract 2.3.1
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · This walk follows employer identities only. Run it once as it is,
#    then add the edge label that follows amounts, and run it again.
#    Change only the line marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
# BFS from the model version, inbound: each node once, with its hops.
WALK = """
MATCH (v:Version {`~id`: $source})
CALL neptune.algo.bfs.levels(v, {traversalDirection: 'inbound',
     // ✏️ add the edge label that follows amounts
     edgeLabels: ['PRODUCED_BY', 'IDENTITY_FROM', 'USED', 'RESTS_ON']})
YIELD node, level
WHERE 'Decision' IN labels(node) OR 'Notice' IN labels(node)
RETURN head(labels(node)) AS kind, node.outcome AS outcome,
       node.day > $fixed AS after_fix, node.day > $recent AS recent,
       count(node) AS found, max(level) AS deepest
"""
show_lineage(WALK, source="version:doc-extract:2.3.1",
             fixed="2026-05-24", recent="2026-09-01")

<details>
<summary>🔑 Reveal solution</summary>

```python
WALK = """
MATCH (v:Version {`~id`: $source})
CALL neptune.algo.bfs.levels(v, {traversalDirection: 'inbound',
     edgeLabels: ['PRODUCED_BY', 'DERIVED_FROM', 'IDENTITY_FROM', 'USED', 'RESTS_ON']})
YIELD node, level
WHERE 'Decision' IN labels(node) OR 'Notice' IN labels(node)
RETURN head(labels(node)) AS kind, node.outcome AS outcome,
       node.day > $fixed AS after_fix, node.day > $recent AS recent,
       count(node) AS found, max(level) AS deepest
"""
```

</details>

Compare the rows. The date filter returns all 24,780 decisions made in those three weeks, but 17,715
of them never used 2.3.1, because their income came from an employer feed. A single hop finds the
7,065 runs that called 2.3.1 and stops on May 24. The walk keeps going: 9,670 decisions rest on what
2.3.1 read, and 2,523 of them came after the fix. Most of those, 1,984, got there through employer
records that kept its reading alive for months.

Option C gets the right answer too, but only barely. Athena's recursive SQL stops at 10 levels, and
this walk needs 9. The lesson is to name what was misread first, and then walk those edges.

**Rule 3: project what connects.**

**Cell 3.3 · See the paths.** This view shows 22 of the walk's decisions, with every node on their
paths back to 2.3.1, one column per hop. Choose **Trace the query** to watch the walk reach each hop in
turn, or choose **amounts** or **identities** to follow one route. The count of decisions reached runs
in the top right. Hover over a node to see what it is and when it was made.

*Under the hood:* `draw_lineage` in `agent/session.py` picks the sample with a fixed seed, asks Neptune
for every relationship on each decision's paths back to 2.3.1, and draws them with
`agent/lineage_view.py`. To read the code, run `draw_lineage??` in a new cell (two question marks: Jupyter shows the function's source).

In [ ]:
# Cell 3.3 · a sample of the walk's decisions, every path back to 2.3.1
# LINEAGE_SAMPLE: a fixed seed, so every account draws the same decisions.
draw_lineage(source="version:doc-extract:2.3.1", sample=LINEAGE_SAMPLE)

The columns are hops from 2.3.1. Decisions made during its three weeks sit close to it. The ringed
decisions came after the fix, farther out along the employer-record chains, and the September decision
is 8 hops away. The thick band follows the busiest record: the employer identity 2.3.1 read from one
applicant's pay stubs, and the 31 decisions from Cell 3.1 that were checked against it.

We now know which decisions rest on 2.3.1. In the last decision, we'll turn to what the agent claims,
and check it against what it actually retrieved.

## Decision 4 · Check now or later

*Guide page: **4 · Check now or later**.*

Decision 2 ended with one question no key could answer: **which past denials cite a policy the agent
never retrieved?** That's the examiner's third ask, the policies the decision applied, turned into a
search across every decision. A decision that cites a policy no search returned claims a rule it never
read.

We can catch that in two places. We can **check now**: the runtime compares each decision's citations
with its retrievals before the recommendation reaches the underwriter. That's a check in the runtime's
code, not a graph query, and it covers every decision from now on. Or we can **investigate later**:
search the decisions already recorded, which is the question Decision 2 couldn't key, and see what
went out because of them. That's where the graph comes in.

**Before you run Cell 4.1, make a guess: under escalate, does the record keep the denial, the
escalation, or both?**

**Cell 4.1 · Check now, in the runtime.** This cell runs a copy of Carlos's application, LA-TEST-00148,
twice in parallel, with a test-only citation that no search returned: once with the check set to
annotate, and once set to escalate. Only test copies accept the injected citation. The cell then plants
a known-bad denial, LA-AUDIT-TEST, directly into DynamoDB, for the next cell to find.

*Under the hood:* `test_copy` writes the copy to the applications table, and `run_with_check` in
`agent/session.py` invokes the runtime twice, with `citation_check` set to `annotate` and to
`escalate`. The check itself is `check_citations` in `agent/runtime_app.py`. Before anything about the
decision is written, it compares the policies the decision cites with the policies its searches
returned. When one is missing, it sets `citation_check`, `ungrounded_citations`, and
`ungrounded_month`, the key of the sparse `ungrounded-index`. To read the notebook side, run
`run_with_check??` in a new cell (two question marks:
Jupyter shows the function's source).

In [ ]:
# Cell 4.1 · Carlos's test copy, annotate and escalate (about a minute)
TEST = test_copy("LA-2026-00148", new_id="LA-TEST-00148")
annotate, escalate = run_with_check(TEST, plant=True)

Same application, same citation, two records. Annotate keeps the outcome `deny` and adds
`citation_check: FAILED`. Escalate records `escalate`, keeps `original_decision: deny`, and writes an
`escalation_reason` for the underwriter. Both carry `ungrounded_month`, so the sparse index from
Decision 2 finds new ones by key. That handles every decision from now on. The ones recorded before the
check existed are the next cell's job.

**Cell 4.2 · ✏️ Investigate later: Decision 2's question 5, in the graph.** No item in DynamoDB holds
"cites a policy it never retrieved": answering it there meant a Scan of the whole table, plus join code
to compare one item's citations with another item's retrievals. In the graph, those are two kinds of
edge on the same decision, so the question is one pattern.

- The first two `MATCH` lines find denials that failed II.A.5.d, and each policy they cite.
- **What you write:** one line that keeps a citation only when the same decision has no
  `RETRIEVED_POLICY` edge to that policy: an edge that *isn't there*. In openCypher, a pattern can be
  used as a condition in `WHERE`.

**When it's right,** we see *✅ Your query matches the reference*, then *Caught LA-AUDIT-TEST*. Without
that line, the query returns every citation; the cell says so and runs the reference query instead, so
the rest of the page still works.

*Under the hood:* `show_ungrounded` in `agent/session.py` sends your query to Neptune Analytics through
`neptune_query`, which calls `ExecuteQuery`. It times the query, so we can set it beside Decision 2's
Scan, and compares its rows with the reference query's. Then it runs one more query to count the
notices these denials sent. To read it, run `show_ungrounded??` in a new cell (two question marks: Jupyter shows the function's source).

In [ ]:
# Cell 4.2 · ✏️ option D: denials citing a policy no search returned
# ═════════════════════════════════════════════════════════════════════════
# ✏️ YOUR TURN · Write the line that keeps only citations no search returned.
#    Change only the line marked ✏️.
# ═════════════════════════════════════════════════════════════════════════
UNGROUNDED = """
MATCH (:Policy {`~id`: $ratio_rule})<-[:VIOLATED]-(d:Decision {outcome: 'deny'})
MATCH (d)-[c:CITED_POLICY]->(p:Policy)
// ✏️ YOUR WHERE NOT HERE
RETURN d.application_id AS application, p.id AS policy,
       c.reference AS cited_as
"""
params["ratio_rule"] = "policy:HUD 4000.1 II.A.5.d"   # the qualifying-ratio rule
show_ungrounded(UNGROUNDED, params)

<details>
<summary>🔑 Reveal solution</summary>

```python
UNGROUNDED = """
MATCH (:Policy {`~id`: $ratio_rule})<-[:VIOLATED]-(d:Decision {outcome: 'deny'})
MATCH (d)-[c:CITED_POLICY]->(p:Policy)
WHERE NOT (d)-[:RETRIEVED_POLICY]->(p)
RETURN d.application_id AS application, p.id AS policy,
       c.reference AS cited_as
"""
```

</details>

The planted case comes first, so we know the query works, followed by your annotate copy. Then come the
past denials, recorded before any check existed, each citing a policy its own searches never returned.
The record had the evidence all along. What DynamoDB priced as a full Scan and join code, the graph
answered with one pattern in about two seconds, and no index was added for it: a graph answers
relationship questions that nobody planned a key for.

The output also counts how many of those denials sent an adverse action notice. Those applicants were
given reasons that rest on a rule the decision never read, and they're who the lender has to contact.

**Cell 4.2a · What went out, in the graph.** This cell takes the planted case and two past denials that
sent a notice, and draws, for each one, the policies it retrieved, the policy it cited without
retrieving, and the adverse action notice that went out.

- The first four lines are Cell 4.2's pattern. `OPTIONAL MATCH (n:Notice)-[:FOR]->(d)` counts each
  decision's notices, and `ORDER BY` keeps the planted case first, then denials that sent a notice.
- `CALL { ... }` runs three walks from each decision, and `UNION` joins them: every policy it retrieved,
  a citation only when no retrieval of the same policy stands beside it, and its notice.

In [ ]:
%%oc -qp params -d {"Decision":"application_id","Policy":"id","Notice":"day"} -l 24 -rel 20
// Cell 4.2a · Ungrounded denials: what each retrieved, what it cited without retrieving, and the notice it sent
MATCH (:Policy {`~id`: $ratio_rule})<-[:VIOLATED]-(d:Decision {outcome: 'deny'})
MATCH (d)-[:CITED_POLICY]->(cited:Policy)
WHERE NOT (d)-[:RETRIEVED_POLICY]->(cited)
WITH DISTINCT d
OPTIONAL MATCH (n:Notice)-[:FOR]->(d)
WITH d, count(n) AS notices
ORDER BY d.application_id = $planted DESC, notices DESC, d.application_id
LIMIT 3
CALL {
  WITH d
  MATCH p = (d)-[:RETRIEVED_POLICY]->(:Policy)
  RETURN p
  UNION
  WITH d
  MATCH p = (d)-[:CITED_POLICY]->(c:Policy)
  WHERE NOT (d)-[:RETRIEVED_POLICY]->(c)
  RETURN p
  UNION
  WITH d
  MATCH p = (:Notice)-[:FOR]->(d)
  RETURN p
}
RETURN p

Each decision sits between its notice, on the left, and its policies, on the right. Most policies hang
off a `RETRIEVED_POLICY` edge. Look for the one connected by `CITED_POLICY` alone: the decision cites it,
but none of its own searches returned it. The notice beside it is what the applicant received. From the
missing edge to the letter that went out is two hops, and it's the same query for any policy and any
decision.

**Rule 4: check what it claims.** Do both: check new decisions as they're made, and search the graph
for the older ones, and for what went out because of them.

## Wrap-up · Four rules

*Guide page: **Wrap-up**.*

We started with an examiner's letter and four asks. Here are the four rules that answer them:

1. **Record what it read.** Write down what the agent was given, separately from what it said, once,
   as it decides. That answers asks 1 to 3 the same way every time.
2. **Key what you know.** Give each question you can name a key, an index, or a count kept in the same
   transaction as the decision. That answers ask 4, and it's what keeps asks 1 to 3 cheap.
3. **Project what connects.** Put what decisions share, and what each result was built from, in the
   graph, and leave the steps in DynamoDB. That covers asks 1 and 2 across decisions.
4. **Check what it claims.** Compare citations with retrievals before a recommendation reaches the
   underwriter, and search the graph for the ones from before. That covers asks 1 and 3 on every
   decision.

Now go to **Wrap-up** in the guide. If you have time left, the guide's **Finished early?** page walks
through two more notebooks, in this same folder: **02 Go deeper.ipynb**, where we write the code this
notebook ran for us, and **03 Kiro.ipynb**, where another AI agent answers the examiner's letter using
your queries.